- Prim's Algorithm Visually Explained | Minimum Spanning Tree ByteQuest: https://youtu.be/EHRqQBlZAtU?si=mcO8EZijyHpcuwAk

  other
- Prim's Minimum Spanning Tree Algorithm | Graph Theory by WilliamFiset: https://youtu.be/jsmMtJpPnhU?si=y0QJaMpfCl4G6N-k

In [3]:
## if your graph is dense the use adjascency matrix??


#### 🌲 Minimum Spanning Trees: Prim's Algorithm Logic

Prim's algorithm is an elegant **greedy optimization algorithm** that extracts an acyclic subset of edges connecting every vertex in an undirected graph while minimizing the total cumulative edge weight cost.

##### 1. Defining the Safe Edge Condition
The algorithm expands outward from a single starting vertex, building a single tree layer-by-layer. To ensure that the graph remains a true tree and never creates a closed circular loop, it relies on a strict definition of a **Safe Edge**:

> **"A safe edge is any edge that connects a vertex already captured inside the spanning tree to an unvisited vertex currently outside the tree bounds."**

```python
# The explicit constraint check inside Prim's loop iteration
if next_vertex_is_in_queue and edge_weight < next_vertex.distance:
```

##### 2. Key Paradigm Shift: Prim vs. Dijkstra

While Prim's algorithm and Dijkstra's algorithm look nearly identical on a code level—both utilizing a binary min-heap priority queue to choose the next node—their underlying optimization intents are fundamentally different:

* **Dijkstra's Algorithm (Global Routing):** Minimizes the *cumulative global cost* all the way from the starting source back to the target destination. Its edge relaxation equation continuously aggregates weights: `new_distance = current.distance + weight(u, v)`.
* **Prim's Algorithm (Local Network Cost):** Minimizes the *immediate infrastructure cost* of adding a new node to the existing network cluster, completely ignoring how far that node is from the initial root node. Its edge relaxation equation evaluates only the raw local weight: `new_distance = weight(u, v)`.


In [15]:
import sys

# =====================================================================
# 1. GRAPH FOUNDATION ENTITIES WITH WEIGHT CONTROLS
# =====================================================================

class Vertex:
    def __init__(self, key):
        self.id = key
        self.neighbors = {}  # Format: {neighbor_vertex_object: edge_weight}
        self.distance = sys.maxsize  # Represents the cheapest connection cost to the tree
        self.previous = None         # Predecessor parent node linking it to the tree

    def set_neighbor(self, nbr, weight=0):
        self.neighbors[nbr] = weight

    def get_neighbors(self):
        return self.neighbors.keys()

    def get_key(self):
        return self.id

    def get_weight(self, nbr):
        return self.neighbors[nbr]

    def __repr__(self):
        return f"Vertex({self.id}, key_cost={self.distance if self.distance != sys.maxsize else 'inf'})"


class Graph:
    def __init__(self):
        self.vertices = {}

    def set_vertex(self, key):
        if key not in self.vertices:
            self.vertices[key] = Vertex(key)
        return self.vertices[key]

    def get_vertex(self, key):
        return self.vertices.get(key, None)

    def add_edge(self, from_key, to_key, weight=0):
        if from_key not in self.vertices:
            self.set_vertex(from_key)
        if to_key not in self.vertices:
            self.set_vertex(to_key)
        # Undirected graph connections
        self.vertices[from_key].set_neighbor(self.vertices[to_key], weight)
        self.vertices[to_key].set_neighbor(self.vertices[from_key], weight)

    def __iter__(self):
        return iter(self.vertices.values())


# =====================================================================
# 2. PRIORITY QUEUE ADT (With Key Membership Check Fix)
# =====================================================================

class PriorityQueue:
    def __init__(self):
        self.heap_list = [0]  # 1-indexed placeholder
        self.current_size = 0

    def _perc_up(self, i):
        while i // 2 > 0:
            if self.heap_list[i] < self.heap_list[i // 2]:
                self.heap_list[i], self.heap_list[i // 2] = self.heap_list[i // 2], self.heap_list[i]
            i = i // 2

    def _perc_down(self, i):
        while (i * 2) <= self.current_size:
            mc = self._min_child(i)
            if self.heap_list[i] > self.heap_list[mc]:
                self.heap_list[i], self.heap_list[mc] = self.heap_list[mc], self.heap_list[i]
            i = mc

    def _min_child(self, i):
        if i * 2 + 1 > self.current_size:
            return i * 2
        else:
            if self.heap_list[i * 2] < self.heap_list[i * 2 + 1]:
                return i * 2
            return i * 2 + 1

    def dequeue_min(self):
        ret_val = self.heap_list[1]
        self.heap_list[1] = self.heap_list[self.current_size]
        self.current_size -= 1
        self.heap_list.pop()
        self._perc_down(1)
        return ret_val

    def change_priority(self, key, new_priority):
        for idx in range(1, self.current_size + 1):
            if self.heap_list[idx][1] == key:
                self.heap_list[idx] = (new_priority, key)
                self._perc_up(idx)
                break

    def __contains__(self, key):
        """
        ✨ CRUCIAL FIX: Checks if the vertex string key exists inside 
        any of the heap tuples, ignoring whatever the current distance priority is.
        """
        for idx in range(1, self.current_size + 1):
            if self.heap_list[idx][1] == key:
                return True
        return False

    def is_empty(self):
        return self.current_size == 0

    def build_queue(self, initial_tuples):
        self.current_size = len(initial_tuples)
        self.heap_list = [0] + list(initial_tuples)
        i = self.current_size // 2
        while i > 0:
            self._perc_down(i)
            i -= 1


# =====================================================================
# 3. PRIM'S MINIMUM SPANNING TREE ENGINE
# =====================================================================

def prim(graph, start_vertex):
    start_vertex.distance = 0
    pq = PriorityQueue()
    initial_elements = [(v.distance, v.get_key()) for v in graph]
    pq.build_queue(initial_elements)

    while not pq.is_empty():
        current_dist, current_key = pq.dequeue_min()
        current_vertex = graph.get_vertex(current_key)
        
        for next_vertex in current_vertex.get_neighbors():
            edge_weight = current_vertex.get_weight(next_vertex)
            
            # ✨ FIXED LINE: Uses the custom __contains__ to check membership safely by key string name
            if next_vertex.get_key() in pq and edge_weight < next_vertex.distance:
                next_vertex.distance = edge_weight
                next_vertex.previous = current_vertex
                pq.change_priority(next_vertex.get_key(), edge_weight)


# =====================================================================
# 4. VERIFICATION SYSTEM DRIVER 
# =====================================================================

broadcast_network = Graph()

broadcast_network.add_edge("A", "B", 2)
broadcast_network.add_edge("A", "C", 3)
broadcast_network.add_edge("B", "C", 1)
broadcast_network.add_edge("B", "D", 1)
broadcast_network.add_edge("B", "E", 4)
broadcast_network.add_edge("C", "F", 5)
broadcast_network.add_edge("D", "E", 6)
broadcast_network.add_edge("E", "F", 1)
broadcast_network.add_edge("F", "G", 1)  # Connecting node G

print("=====================================================================")
print("🌲 EXECUTING PRIM'S MINIMUM SPANNING TREE SCHEDULER")
print("=====================================================================\n")

start_node = broadcast_network.get_vertex("A")
prim(broadcast_network, start_node)

print("🏁 IDENTIFIED MINIMUM SPANNING TREE EDGES:")
print("-" * 55)
total_mst_cost = 0

for key in sorted(broadcast_network.vertices.keys()):
    v = broadcast_network.get_vertex(key)
    if v.previous:
        print(f" 🎯 Link Edge: {v.previous.get_key()} ──({v.distance})──► {v.get_key()}")
        total_mst_cost += v.distance

print("-" * 55)
print(f"🎉 Accumulated Minimum Spanning Tree Weight Cost: {total_mst_cost}")
print("=====================================================================")


🌲 EXECUTING PRIM'S MINIMUM SPANNING TREE SCHEDULER

🏁 IDENTIFIED MINIMUM SPANNING TREE EDGES:
-------------------------------------------------------
 🎯 Link Edge: A ──(2)──► B
 🎯 Link Edge: B ──(1)──► C
 🎯 Link Edge: B ──(1)──► D
 🎯 Link Edge: B ──(4)──► E
 🎯 Link Edge: E ──(1)──► F
 🎯 Link Edge: F ──(1)──► G
-------------------------------------------------------
🎉 Accumulated Minimum Spanning Tree Weight Cost: 10


#### Playground

In [13]:
import sys

# =====================================================================
# 1. GRAPH FOUNDATION ENTITIES WITH WEIGHT CONTROLS
# =====================================================================

class Vertex:
    def __init__(self, key):
        """Initializes a node tracking connections and spanning keys."""
        self.id = key
        self.neighbors = {}  # Format: {neighbor_vertex_object: edge_weight}
        
        # Prim's path tracking state attributes
        self.distance = sys.maxsize  # Represents the cheapest connection cost to the tree
        self.previous = None         # Predecessor parent node linking it to the tree

    def set_neighbor(self, nbr, weight=0):
        """Registers a bidirectional weighted edge configuration."""
        self.neighbors[nbr] = weight

    def get_neighbors(self):
        """Returns adjacent vertex objects."""
        return self.neighbors.keys()

    def get_key(self):
        return self.id

    def get_weight(self, nbr):
        """Returns the edge weight linking this node to the neighbor."""
        return self.neighbors[nbr]

    def __repr__(self):
        return f"Vertex({self.id}, key_cost={self.distance if self.distance != sys.maxsize else 'inf'})"


class Graph:
    def __init__(self):
        self.vertices = {}

    def set_vertex(self, key):
        if key not in self.vertices:
            self.vertices[key] = Vertex(key)
        return self.vertices[key]

    def get_vertex(self, key):
        return self.vertices.get(key, None)

    def add_edge(self, from_key, to_key, weight=0):
        """Creates an undirected network link with a specified cost weight."""
        if from_key not in self.vertices:
            self.set_vertex(from_key)
        if to_key not in self.vertices:
            self.set_vertex(to_key)
        # Minimum Spanning Trees are evaluated on undirected graphs; connect both ways
        self.vertices[from_key].set_neighbor(self.vertices[to_key], weight)
        self.vertices[to_key].set_neighbor(self.vertices[from_key], weight)

    def __iter__(self):
        return iter(self.vertices.values())


# =====================================================================
# 2. PRIORITY QUEUE ADT (Binary Min-Heap Component Framework)
# =====================================================================

class PriorityQueue:
    def __init__(self):
        self.heap_list = [0]  # 1-indexed placeholder
        self.current_size = 0

    def _perc_up(self, i):
        while i // 2 > 0:
            if self.heap_list[i] < self.heap_list[i // 2]:
                self.heap_list[i], self.heap_list[i // 2] = self.heap_list[i // 2], self.heap_list[i]
            i = i // 2

    def _perc_down(self, i):
        while (i * 2) <= self.current_size:
            mc = self._min_child(i)
            if self.heap_list[i] > self.heap_list[mc]:
                self.heap_list[i], self.heap_list[mc] = self.heap_list[mc], self.heap_list[i]
            i = mc

    def _min_child(self, i):
        if i * 2 + 1 > self.current_size:
            return i * 2
        else:
            if self.heap_list[i * 2] < self.heap_list[i * 2 + 1]:
                return i * 2
            return i * 2 + 1

    def dequeue_min(self):
        ret_val = self.heap_list[1]
        self.heap_list[1] = self.heap_list[self.current_size]
        self.current_size -= 1
        self.heap_list.pop()
        self._perc_down(1)
        return ret_val

    def change_priority(self, key, new_priority):
        """Locates the element in the heap and updates its internal sorting priority."""
        for idx in range(1, self.current_size + 1):
            if self.heap_list[idx][1] == key:
                self.heap_list[idx] = (new_priority, key)
                self._perc_up(idx)
                break

    def is_empty(self):
        return self.current_size == 0

    def build_queue(self, initial_tuples):
        self.current_size = len(initial_tuples)
        self.heap_list = [0] + list(initial_tuples)
        i = self.current_size // 2
        while i > 0:
            self._perc_down(i)
            i -= 1


# =====================================================================
# 3. PRIM'S MINIMUM SPANNING TREE ENGINE
# =====================================================================

def prim(graph, start_vertex):
    """
    Listing 7.22.3: Prim's algorithm generating a minimum spanning tree forest
    by always picking the cheapest safe edge connecting to a new unvisited vertex.
    """
    # 1. Initialize the root starting position entry to zero cost
    start_vertex.distance = 0
    
    # 2. Build the priority queue holding all graph node parameters
    pq = PriorityQueue()
    initial_elements = [(v.distance, v.get_key()) for v in graph]
    pq.build_queue(initial_elements)

    while not pq.is_empty():
        # Dequeue the element closest to the existing spanning tree structure boundaries
        current_dist, current_key = pq.dequeue_min()
        current_vertex = graph.get_vertex(current_key)
        
        # 3. Iterate over neighbors to identify safe bridge expansions
        for next_vertex in current_vertex.get_neighbors():
            # Prim's Rule: Evaluate ONLY the direct raw edge weight linking to this neighbor
            edge_weight = current_vertex.get_weight(next_vertex)
            
            # If the neighbor is still in the priority queue (not finalized) AND 
            # this new direct edge connection is cheaper than its prior alternative cost
            if (edge_weight, next_vertex.get_key()) in pq.heap_list and edge_weight < next_vertex.distance:
                # Update tracking distance variables and reset predecessor links
                next_vertex.distance = edge_weight
                next_vertex.previous = current_vertex
                
                # Signal priority heap shift
                pq.change_priority(next_vertex.get_key(), edge_weight)


# =====================================================================
# 4. VERIFICATION SYSTEM DRIVER (Graph Mirroring Figures 11-17)
# =====================================================================

# Assembling the sample broadcast tree described in your text trace walkthrough
broadcast_network = Graph()

broadcast_network.add_edge("A", "B", 2)
broadcast_network.add_edge("A", "C", 3)

broadcast_network.add_edge("B", "C", 1)
broadcast_network.add_edge("B", "D", 1)
broadcast_network.add_edge("B", "E", 4)

broadcast_network.add_edge("C", "F", 5)

broadcast_network.add_edge("D", "E", 6)

broadcast_network.add_edge("E", "F", 1)

print("=====================================================================")
print("🌲 EXECUTING PRIM'S MINIMUM SPANNING TREE SCHEDULER")
print("=====================================================================\n")

# Run calculation starting at node 'A'
start_node = broadcast_network.get_vertex("A")
prim(broadcast_network, start_node)

print("🏁 IDENTIFIED MINIMUM SPANNING TREE EDGES:")
print("-" * 55)
total_mst_cost = 0

for key in sorted(broadcast_network.vertices.keys()):
    v = broadcast_network.get_vertex(key)
    if v.previous:
        print(f" 🎯 Link Edge: {v.previous.get_key()} ──({v.distance})──► {v.get_key()}")
        total_mst_cost += v.distance

print("-" * 55)
print(f"🎉 Accumulated Minimum Spanning Tree Weight Cost: {total_mst_cost}")
print("=====================================================================")


🌲 EXECUTING PRIM'S MINIMUM SPANNING TREE SCHEDULER

🏁 IDENTIFIED MINIMUM SPANNING TREE EDGES:
-------------------------------------------------------
-------------------------------------------------------
🎉 Accumulated Minimum Spanning Tree Weight Cost: 0


In [14]:
# =====================================================================
# 4. VERIFICATION SYSTEM DRIVER (Graph Mirroring Figures 11-17)
# =====================================================================

broadcast_network = Graph()

broadcast_network.add_edge("A", "B", 2)
broadcast_network.add_edge("A", "C", 3)
broadcast_network.add_edge("B", "C", 1)
broadcast_network.add_edge("B", "D", 1)
broadcast_network.add_edge("B", "E", 4)
broadcast_network.add_edge("C", "F", 5)
broadcast_network.add_edge("D", "E", 6)
broadcast_network.add_edge("E", "F", 1)

# ──► ADD THIS MISSING LINE TO CONNECT NODE G TO THE NETWORK ◄──
broadcast_network.add_edge("F", "G", 1)  # Matching standard textbook configurations


print("=====================================================================")
print("🌲 EXECUTING PRIM'S MINIMUM SPANNING TREE SCHEDULER")
print("=====================================================================\n")

# Run calculation starting at node 'A'
start_node = broadcast_network.get_vertex("A")
prim(broadcast_network, start_node)

print("🏁 IDENTIFIED MINIMUM SPANNING TREE EDGES:")
print("-" * 55)
total_mst_cost = 0

for key in sorted(broadcast_network.vertices.keys()):
    v = broadcast_network.get_vertex(key)
    if v.previous:
        print(f" 🎯 Link Edge: {v.previous.get_key()} ──({v.distance})──► {v.get_key()}")
        total_mst_cost += v.distance

print("-" * 55)
print(f"🎉 Accumulated Minimum Spanning Tree Weight Cost: {total_mst_cost}")
print("=====================================================================")


🌲 EXECUTING PRIM'S MINIMUM SPANNING TREE SCHEDULER

🏁 IDENTIFIED MINIMUM SPANNING TREE EDGES:
-------------------------------------------------------
-------------------------------------------------------
🎉 Accumulated Minimum Spanning Tree Weight Cost: 0
